[Open in Colab](https://colab.research.google.com/github/BoydVonstocker/Introduction-to-Quantum-Computing/blob/main/Notebks/04%20-%20Measurement_and_Bloch_Sphere.ipynb)


# Notebook 04 — Measurement and the Bloch Sphere

Two pictures dominate introductory quantum computing:

1. the **Bloch sphere**, which draws any pure single-qubit state as a point on a sphere
2. **projective measurement**, which turns amplitudes into random classical bits


In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_vector, plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator
import numpy as np


## Bloch sphere coordinates

Any pure qubit can be written, up to a global phase,

$$
|\psi\rangle = \cos\frac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}\,|1\rangle.
$$

$(\theta,\varphi)$ are spherical coordinates:

- $\theta = 0$ is $|0\rangle$ (north pole)
- $\theta = \pi$ is $|1\rangle$ (south pole)
- $\theta = \pi/2,\ \varphi = 0$ is $|+\rangle$
- $\theta = \pi/2,\ \varphi = \pi$ is $|-\rangle$
- $\theta = \pi/2,\ \varphi = \pi/2$ is $|+i\rangle$

The Bloch vector is $\vec{r} = (\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta)$.


In [ ]:
# |+> lives on the +x axis of the Bloch sphere
plot_bloch_vector([1, 0, 0], title=r"$|+\rangle$")


In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)   # |+> -> |+i>
plot_bloch_multivector(Statevector.from_instruction(qc), title=r"$S|+\rangle = |+i\rangle$")


## What a computational-basis measurement does

A $Z$-measurement projects onto $|0\rangle$ or $|1\rangle$:

$$
p(0) = |\langle 0|\psi\rangle|^2 = |\alpha|^2, \qquad
p(1) = |\beta|^2.
$$

After the shot, the qubit *is* the observed basis state. Superposition does not survive an unprotected measurement.


In [ ]:
# |psi> = sqrt(3/4)|0> + sqrt(1/4)|1>
qc = QuantumCircuit(1)
qc.ry(2 * np.arccos(np.sqrt(0.75)), 0)
sv = Statevector.from_instruction(qc)
print(sv.probabilities_dict())
plot_bloch_multivector(sv)


In [ ]:
qc_m = QuantumCircuit(1, 1)
qc_m.ry(2 * np.arccos(np.sqrt(0.75)), 0)
qc_m.measure(0, 0)
counts = AerSimulator().run(qc_m, shots=4000).result().get_counts()
plot_histogram(counts)


## Measuring in another basis

To measure in the $X$ basis $\{|+\rangle,|-\rangle\}$, rotate with $H$ first, then measure in $Z$.

$$
H|+\rangle = |0\rangle, \qquad H|-\rangle = |1\rangle.
$$

So a `0` after $H$ means the qubit was $|+\rangle$.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)          # prepare |+>
qc.h(0)          # rotate X-basis -> Z-basis
qc.measure(0, 0)
display(qc.draw("mpl"))
plot_histogram(AerSimulator().run(qc, shots=1024).result().get_counts())


## Shot noise versus exact probabilities

The statevector gives exact Born probabilities. A finite number of shots is a multinomial sample of those probabilities. Increase `shots` and the histogram tightens around the true values.


In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
for shots in (16, 128, 2048):
    counts = AerSimulator().run(qc, shots=shots).result().get_counts()
    print(f"shots={shots:4d}  {counts}")


## Several qubits on the Bloch sphere

`plot_bloch_multivector` draws one sphere per qubit using the *reduced* state of that qubit. For entangled states the arrows shrink, because a single qubit of a Bell pair is mixed. Use a Q-sphere or the full statevector when entanglement matters (see Notebook 03).


In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
plot_bloch_multivector(Statevector.from_instruction(qc), title="Bell pair: local Bloch vectors vanish")


## Exercises

1. Find $(\theta,\varphi)$ for $|-i\rangle = (|0\rangle - i|1\rangle)/\sqrt{2}$ and plot it.
2. Prepare $\cos(\pi/6)|0\rangle + \sin(\pi/6)|1\rangle$ with `ry`, measure 2000 shots, and compare to $\sin^2(\pi/6)$.
3. Measure $|+\rangle$ in the $Z$ basis and in the $X$ basis. Why are the histograms so different?


In [ ]:
# Exercise workspace
